In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings

# Suppress pandas future warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Data Loading and Cleaning
try:
    df = pd.read_csv('India_Trade_Tariff_Data_Cleaned.csv')
except FileNotFoundError:
    print("Error: 'India_Trade_Tariff_Data_Cleaned.csv' not found. Please ensure the file is in the working directory.")
    exit()

# Simulate 2025 data to add one set of 5 rows
if len(df) == 15:  # Original dataset has 15 rows (2022–2024, 5 partners)
    df_2025 = df[df['Year'] == 2024].copy()
    df_2025['Year'] = 2025
    df = pd.concat([df, df_2025], ignore_index=True)  # Now 20 rows

# Verify DataFrame size
num_rows = len(df)
if num_rows != 20:
    print(f"Warning: Expected 20 rows (15 original + 5 for 2025), got {num_rows} rows.")
    # Adjust to 20 rows if needed
    if num_rows > 20:
        df = df.head(20)
    else:
        print("Error: Insufficient rows. Please check dataset.")
        exit()
num_rows = len(df)  # Update num_rows after adjustment

# Add missing columns with simulated values, truncated to num_rows
df['India_GDP_Growth_Percent'] = ([7.0, 7.0, 7.0, 7.0, 7.0,  # 2022
                                   7.2, 7.2, 7.2, 7.2, 7.2,  # 2023
                                   6.5, 6.5, 6.5, 6.5, 6.5,  # 2024
                                   -1.0, 2.0, 6.2, 6.2, 6.2] + [0]*5)[:num_rows]  # 2025
df['INR_USD_Exchange_Rate'] = ([78.6, 78.6, 78.6, 78.6, 78.6,  # 2022
                                82.3, 82.3, 82.3, 82.3, 82.3,  # 2023
                                83.5, 83.5, 83.5, 83.5, 83.5,  # 2024
                                85.0, 85.0, 85.0, 85.0, 85.0] + [85.0]*5)[:num_rows]  # 2025
df['Auto_Tariff_Percent'] = ([2.5, 7.0, 4.5, 6.5, 4.5,  # 2022
                              2.5, 7.0, 4.5, 6.5, 4.5,  # 2023
                              2.5, 7.0, 4.5, 6.5, 4.5,  # 2024
                              20.0, 10.0, 4.5, 6.5, 4.5] + [0]*5)[:num_rows]  # 2025
df['Textile_Tariff_Percent'] = ([3.0, 8.0, 5.0, 7.0, 5.0,  # 2022
                                 3.0, 8.0, 5.0, 7.0, 5.0,  # 2023
                                 3.0, 8.0, 5.0, 7.0, 5.0,  # 2024
                                 20.0, 10.0, 5.0, 7.0, 5.0] + [0]*5)[:num_rows]  # 2025
df['Z_GDP_Growth'] = (df['India_GDP_Growth_Percent'] - df['India_GDP_Growth_Percent'].mean()) / df['India_GDP_Growth_Percent'].std()
df['Z_Exchange_Rate'] = (df['INR_USD_Exchange_Rate'] - df['INR_USD_Exchange_Rate'].mean()) / df['INR_USD_Exchange_Rate'].std()

# Enhance 2025 data
df.loc[df['Year'] == 2025, 'Partner_Tariff_On_India_Percent'] = [20.0 if tp == 'USA' else 10.0 if tp == 'China' else 5.0 for tp in df.loc[df['Year'] == 2025, 'Trade_Partner']]
df.loc[df['Year'] == 2025, 'Export_Value_USD_Billion'] *= [0.85 if tp == 'USA' else 0.90 if tp == 'China' else 1.0 for tp in df.loc[df['Year'] == 2025, 'Trade_Partner']]
df['Log_Export_Value'] = np.log(df['Export_Value_USD_Billion'])
df['Z_Partner_Tariff'] = (df['Partner_Tariff_On_India_Percent'] - df['Partner_Tariff_On_India_Percent'].mean()) / df['Partner_Tariff_On_India_Percent'].std()

# Ensure correct data types
df['Year'] = df['Year'].astype(int)
df['Trade_Partner'] = df['Trade_Partner'].astype('category')
numeric_cols = ['Export_Value_USD_Billion', 'Log_Export_Value', 'Auto_Tariff_Percent',
                'Textile_Tariff_Percent', 'India_GDP_Growth_Percent', 'INR_USD_Exchange_Rate',
                'Z_GDP_Growth', 'Z_Exchange_Rate', 'Z_Partner_Tariff']
for col in numeric_cols:
    if col not in df.columns:
        print(f"Error: Column {col} missing from DataFrame.")
        exit()
df[numeric_cols] = df[numeric_cols].astype(float)

# Verify column presence
required_cols = ['Year', 'Trade_Partner', 'Export_Value_USD_Billion', 'Log_Export_Value',
                 'Auto_Tariff_Percent', 'Textile_Tariff_Percent', 'India_GDP_Growth_Percent',
                 'INR_USD_Exchange_Rate', 'Z_GDP_Growth', 'Z_Exchange_Rate', 'Z_Partner_Tariff']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    print(f"Error: Missing columns: {missing_cols}")
    exit()

# 2. Fit OLS Model for GDP Growth
exrate_mean = df['INR_USD_Exchange_Rate'].mean()
exrate_std = df['INR_USD_Exchange_Rate'].std()
# In the first half, ensure Z_GDP_Growth is defined
df['Z_GDP_Growth'] = (df['India_GDP_Growth_Percent'] - df['India_GDP_Growth_Percent'].mean()) / df['India_GDP_Growth_Percent'].std()

# Update OLS model formula
# In the first half, no need for Z_GDP_Growth or Log_Export_Value
# Update OLS model formula
ols_formula = 'India_GDP_Growth_Percent ~ Partner_Tariff_On_India_Percent + Z_Exchange_Rate'
ols_model = smf.ols(ols_formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nOLS Regression Results (GDP Growth %):\n")
print(ols_model.summary())
# VIF Analysis
X = df[['Log_Export_Value', 'Auto_Tariff_Percent', 'Z_GDP_Growth', 'Z_Exchange_Rate']].copy()
X['Intercept'] = 1
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVariance Inflation Factors:\n", vif_data)

# 3. Predictive Function
BASE_GDP = 21064660  # Base GDP in USD million ($21.06T)
def predict_gdp_growth(country, log_export, auto_tariff_usa, auto_tariff_china, auto_tariff_uae, auto_tariff_russia, auto_tariff_saudi,
                       textile_tariff_usa, textile_tariff_china, textile_tariff_uae, textile_tariff_russia, textile_tariff_saudi, gdp_growth, exchange_rate):
    z_gdp = (gdp_growth - df['India_GDP_Growth_Percent'].mean()) / df['India_GDP_Growth_Percent'].std()
    z_exrate = (exchange_rate - exrate_mean) / exrate_std
    # Map country to tariff values
    auto_tariff_map = {
        'USA': auto_tariff_usa,
        'China': auto_tariff_china,
        'UAE': auto_tariff_uae,
        'Russia': auto_tariff_russia,
        'Saudi Arabia': auto_tariff_saudi
    }
    textile_tariff_map = {
        'USA': textile_tariff_usa,
        'China': textile_tariff_china,
        'UAE': textile_tariff_uae,
        'Russia': textile_tariff_russia,
        'Saudi Arabia': textile_tariff_saudi
    }
    auto_tariff = auto_tariff_map[country]
    input_data = pd.DataFrame({
        'Intercept': [1],
        'Log_Export_Value': [log_export],
        'Auto_Tariff_Percent': [auto_tariff],
        'Z_GDP_Growth': [z_gdp],
        'Z_Exchange_Rate': [z_exrate]
    })
    gdp_growth_pred = ols_model.predict(input_data)[0]  # Predicted growth in %
    new_gdp = (BASE_GDP * (gdp_growth_pred / 100)) + BASE_GDP  # [(base_gdp * growth%) + base_gdp]
    print(f"Country: {country}")
    print(f"Predicted GDP Growth: {gdp_growth_pred:.2f}%")
    print(f"New GDP: ${new_gdp:,.2f} million")
    return gdp_growth_pred, new_gdp




OLS Regression Results (GDP Growth %):

                               OLS Regression Results                               
Dep. Variable:     India_GDP_Growth_Percent   R-squared:                       0.820
Model:                                  OLS   Adj. R-squared:                  0.799
Method:                       Least Squares   F-statistic:                     891.6
Date:                      Mon, 28 Apr 2025   Prob (F-statistic):           5.01e-06
Time:                              14:13:27   Log-Likelihood:                -24.854
No. Observations:                        20   AIC:                             55.71
Df Residuals:                            17   BIC:                             58.70
Df Model:                                 2                                         
Covariance Type:                    cluster                                         
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------

In [ ]:
# VIF Analysis
X = df[['Partner_Tariff_On_India_Percent', 'Z_Exchange_Rate']].copy()
X['Intercept'] = 1
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVariance Inflation Factors:\n", vif_data)

# 3. Predictive Function
BASE_GDP = 3500000  # Base GDP in USD million ($3.5T)
def predict_gdp_growth(total_tariff_usa, total_tariff_china, total_tariff_uae, total_tariff_russia, total_tariff_saudi, exchange_rate):
    # Calculate mean tariff
    tariffs = [total_tariff_usa, total_tariff_china, total_tariff_uae, total_tariff_russia, total_tariff_saudi]
    mean_tariff = np.mean(tariffs)
    # Calculate standardization parameters within the function
    exrate_mean = df['INR_USD_Exchange_Rate'].mean()
    exrate_std = df['INR_USD_Exchange_Rate'].std()
    z_exrate = (exchange_rate - exrate_mean) / exrate_std
    input_data = pd.DataFrame({
        'Intercept': [1],
        'Partner_Tariff_On_India_Percent': [mean_tariff],
        'Z_Exchange_Rate': [z_exrate]
    })
    gdp_growth_pred = ols_model.predict(input_data)[0]  # Predicted growth in %
    new_gdp = (BASE_GDP * (gdp_growth_pred/100) + BASE_GDP ) # [(base_gdp * growth%) + base_gdp]
    print(f"Mean Total Tariff: {mean_tariff:.2f}%")
    print(f"Predicted India GDP Growth: {gdp_growth_pred:.2f}%")
    print(f"New India GDP: ${new_gdp:,.2f} million")
    return gdp_growth_pred, new_gdp

# 4. Slider Interface
total_tariff_usa_slider = widgets.FloatSlider(value=3.3, min=0, max=25, step=0.1, description=' USA (%):')
total_tariff_china_slider = widgets.FloatSlider(value=7.5, min=0, max=25, step=0.1, description=' China (%):')
total_tariff_uae_slider = widgets.FloatSlider(value=5.0, min=0, max=25, step=0.1, description=' UAE (%):')
total_tariff_russia_slider = widgets.FloatSlider(value=7.0, min=0, max=25, step=0.1, description=' Russia (%):')
total_tariff_saudi_slider = widgets.FloatSlider(value=5.0, min=0, max=25, step=0.1, description=' Saudi (%):')
exrate_slider = widgets.FloatSlider(value=83.5, min=75, max=90, step=0.1, description='INR/USD Rate:')

# Interactive prediction
print("Predict India GDP Growth")
gdp_output = widgets.interactive_output(predict_gdp_growth, {
    'total_tariff_usa': total_tariff_usa_slider,
    'total_tariff_china': total_tariff_china_slider,
    'total_tariff_uae': total_tariff_uae_slider,
    'total_tariff_russia': total_tariff_russia_slider,
    'total_tariff_saudi': total_tariff_saudi_slider,
    'exchange_rate': exrate_slider
})
display(widgets.VBox([total_tariff_usa_slider, total_tariff_china_slider, total_tariff_uae_slider,
                      total_tariff_russia_slider, total_tariff_saudi_slider, exrate_slider, gdp_output]))

# Save model parameters
model_params = ols_model.params.to_dict()
with open('gdp_growth_model_parameters.txt', 'w') as f:
    f.write(str(model_params))
print("\nModel parameters saved as 'gdp_growth_model_parameters.txt'")


Variance Inflation Factors:
                           Variable       VIF
0  Partner_Tariff_On_India_Percent  1.080421
1                  Z_Exchange_Rate  1.080421
2                        Intercept  4.575472
Predict India GDP Growth



Model parameters saved as 'gdp_growth_model_parameters.txt'


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import display, clear_output

# VIF Analysis
X = df[['Partner_Tariff_On_India_Percent', 'Z_Exchange_Rate']].copy()
X['Intercept'] = 1
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVariance Inflation Factors:\n", vif_data)

# Predictive and Visualization Function
BASE_GDP = 3500000  # Base GDP in USD million ($3.5T)
def predict_and_visualize(total_tariff_usa, total_tariff_china, total_tariff_uae, total_tariff_russia, total_tariff_saudi, exchange_rate):
    # Clear previous output
    clear_output(wait=True)

    # Calculate mean tariff
    tariffs = [total_tariff_usa, total_tariff_china, total_tariff_uae, total_tariff_russia, total_tariff_saudi]
    mean_tariff = np.mean(tariffs)
    # Calculate standardization parameters
    exrate_mean = df['INR_USD_Exchange_Rate'].mean()
    exrate_std = df['INR_USD_Exchange_Rate'].std()
    z_exrate = (exchange_rate - exrate_mean) / exrate_std
    input_data = pd.DataFrame({
        'Intercept': [1],
        'Partner_Tariff_On_India_Percent': [mean_tariff],
        'Z_Exchange_Rate': [z_exrate]
    })
    gdp_growth_pred = ols_model.predict(input_data)[0]  # Predicted growth in %
    new_gdp = (BASE_GDP * (gdp_growth_pred / 100)) + BASE_GDP  # [(base_gdp * growth%) + base_gdp]

    # Table 1: OLS Coefficients and VIFs
    coef_table = pd.DataFrame({
        'Coefficient': ols_model.params,
        'P-Value': ols_model.pvalues,
        'Std Error': ols_model.bse
    }).round(4)
    coef_table = coef_table.merge(vif_data, left_index=True, right_on='Variable', how='left').fillna('-')
    print("\nTable 1: OLS Coefficients and VIFs\n", coef_table)

    # Table 2: Prediction Results
    pred_table = pd.DataFrame({
        'Mean Tariff (%)': [mean_tariff],
        'Exchange Rate (INR/USD)': [exchange_rate],
        'Predicted GDP Growth (%)': [gdp_growth_pred],
        'New GDP ($M)': [new_gdp]
    }).round(2)
    print("\nTable 2: Prediction Results\n", pred_table)

    # Visualizations
    # Chart 1: Bar Plot - OLS Coefficients
    plt.figure(figsize=(8, 6))
    coef = ols_model.params[1:]  # Exclude intercept
    errors = ols_model.bse[1:]
    plt.bar(coef.index, coef, yerr=errors, color='skyblue', capsize=5)
    plt.axhline(0, color='black', linestyle='--')
    plt.xlabel('Predictors')
    plt.ylabel('Coefficient Value')
    plt.title('OLS Coefficients for GDP Growth Prediction')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('coefficient_plot.png')
    plt.show()
    plt.close()

    # Chart 2: Sensitivity Plot - GDP Growth vs. Mean Tariff
    tariffs_range = np.linspace(0, 25, 50)
    growth_preds_tariff = []
    for tariff in tariffs_range:
        input_data_tariff = pd.DataFrame({
            'Intercept': [1],
            'Partner_Tariff_On_India_Percent': [tariff],
            'Z_Exchange_Rate': [z_exrate]
        })
        growth_preds_tariff.append(ols_model.predict(input_data_tariff)[0])
    plt.figure(figsize=(8, 6))
    plt.plot(tariffs_range, growth_preds_tariff, color='blue')
    plt.axhline(0, color='black', linestyle='--')
    plt.xlabel('Mean Tariff (%)')
    plt.ylabel('Predicted GDP Growth (%)')
    plt.title('Sensitivity of GDP Growth to Mean Tariff')
    plt.tight_layout()
    plt.savefig('sensitivity_tariff.png')
    plt.show()
    plt.close()

    # Chart 3: Sensitivity Plot - GDP Growth vs. Exchange Rate
    exrate_range = np.linspace(75, 90, 50)
    growth_preds_exrate = []
    for exrate in exrate_range:
        z_exrate_temp = (exrate - exrate_mean) / exrate_std
        input_data_exrate = pd.DataFrame({
            'Intercept': [1],
            'Partner_Tariff_On_India_Percent': [mean_tariff],
            'Z_Exchange_Rate': [z_exrate_temp]
        })
        growth_preds_exrate.append(ols_model.predict(input_data_exrate)[0])
    plt.figure(figsize=(8, 6))
    plt.plot(exrate_range, growth_preds_exrate, color='green')
    plt.axhline(0, color='black', linestyle='--')
    plt.xlabel('INR/USD Exchange Rate')
    plt.ylabel('Predicted GDP Growth (%)')
    plt.title('Sensitivity of GDP Growth to Exchange Rate')
    plt.tight_layout()
    plt.savefig('sensitivity_exrate.png')
    plt.show()
    plt.close()

    # Chart 4: Heatmap of GDP Growth vs. Tariff and Exchange Rate
    tariff_grid, exrate_grid = np.meshgrid(np.linspace(0, 25, 20), np.linspace(75, 90, 20))
    growth_grid = np.zeros_like(tariff_grid)
    for i in range(tariff_grid.shape[0]):
        for j in range(tariff_grid.shape[1]):
            z_exrate_temp = (exrate_grid[i, j] - exrate_mean) / exrate_std
            input_data_grid = pd.DataFrame({
                'Intercept': [1],
                'Partner_Tariff_On_India_Percent': [tariff_grid[i, j]],
                'Z_Exchange_Rate': [z_exrate_temp]
            })
            growth_grid[i, j] = ols_model.predict(input_data_grid)[0]
    plt.figure(figsize=(10, 8))
    sns.heatmap(growth_grid, xticklabels=np.round(tariff_grid[0], 1), yticklabels=np.round(exrate_grid[:, 0], 1),
                cmap='RdYlGn', annot=True, fmt='.1f')
    plt.xlabel('Mean Tariff (%)')
    plt.ylabel('INR/USD Exchange Rate')
    plt.title('Predicted GDP Growth Heatmap')
    plt.tight_layout()
    plt.savefig('heatmap_gdp_growth.png')
    plt.show()
    plt.close()

    # Chart 5: Time Series Plot - Actual vs. Predicted GDP Growth
    df['Predicted_GDP_Growth'] = ols_model.predict(df)
    plt.figure(figsize=(10, 6))
    for partner in df['Trade_Partner'].unique():
        partner_data = df[df['Trade_Partner'] == partner]
        plt.plot(partner_data['Year'], partner_data['India_GDP_Growth_Percent'], marker='o', label=f'{partner} Actual')
        plt.plot(partner_data['Year'], partner_data['Predicted_GDP_Growth'], marker='x', linestyle='--', label=f'{partner} Predicted')
    plt.xlabel('Year')
    plt.ylabel('GDP Growth (%)')
    plt.title('Actual vs. Predicted GDP Growth by Trade Partner')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('time_series_gdp_growth.png')
    plt.show()
    plt.close()

    # Chart 6: Scatter Plot - Predicted GDP Growth vs. Mean Tariff
    tariff_range_scatter = np.linspace(0, 25, 20)
    exrate_values = [75, 80, 85, 90]
    plt.figure(figsize=(8, 6))
    for exrate in exrate_values:
        z_exrate_temp = (exrate - exrate_mean) / exrate_std
        growth_preds = []
        for tariff in tariff_range_scatter:
            input_data_scatter = pd.DataFrame({
                'Intercept': [1],
                'Partner_Tariff_On_India_Percent': [tariff],
                'Z_Exchange_Rate': [z_exrate_temp]
            })
            growth_preds.append(ols_model.predict(input_data_scatter)[0])
        plt.scatter(tariff_range_scatter, growth_preds, label=f'INR/USD={exrate}')
    plt.axhline(0, color='black', linestyle='--')
    plt.xlabel('Mean Tariff (%)')
    plt.ylabel('Predicted GDP Growth (%)')
    plt.title('Predicted GDP Growth vs. Mean Tariff by Exchange Rate')
    plt.legend()
    plt.tight_layout()
    plt.savefig('scatter_tariff_exrate.png')
    plt.show()
    plt.close()

# Slider Interface
total_tariff_usa_slider = widgets.FloatSlider(value=3.3, min=0, max=25, step=0.1, description='Total Tariff USA (%):')
total_tariff_china_slider = widgets.FloatSlider(value=7.5, min=0, max=25, step=0.1, description='Total Tariff China (%):')
total_tariff_uae_slider = widgets.FloatSlider(value=5.0, min=0, max=25, step=0.1, description='Total Tariff UAE (%):')
total_tariff_russia_slider = widgets.FloatSlider(value=7.0, min=0, max=25, step=0.1, description='Total Tariff Russia (%):')
total_tariff_saudi_slider = widgets.FloatSlider(value=5.0, min=0, max=25, step=0.1, description='Total Tariff Saudi (%):')
exrate_slider = widgets.FloatSlider(value=83.5, min=75, max=90, step=0.1, description='INR/USD Rate:')

# Interactive prediction
print("Predict India GDP Growth")
gdp_output = widgets.interactive_output(predict_and_visualize, {
    'total_tariff_usa': total_tariff_usa_slider,
    'total_tariff_china': total_tariff_china_slider,
    'total_tariff_uae': total_tariff_uae_slider,
    'total_tariff_russia': total_tariff_russia_slider,
    'total_tariff_saudi': total_tariff_saudi_slider,
    'exchange_rate': exrate_slider
})
display(widgets.VBox([total_tariff_usa_slider, total_tariff_china_slider, total_tariff_uae_slider,
                      total_tariff_russia_slider, total_tariff_saudi_slider, exrate_slider, gdp_output]))


Variance Inflation Factors:
                           Variable       VIF
0  Partner_Tariff_On_India_Percent  1.080421
1                  Z_Exchange_Rate  1.080421
2                        Intercept  4.575472
Predict India GDP Growth
